In [7]:
from pulse_uduu_longtime import *

# Example usage
if __name__ == "__main__":
    data = run_uduu_long(
        pulse_amplitude=0.5,
        pulse_width_ns=200.0,
        u_to_d_delay=200.0,
        d_to_u_delay_s=1,  # in s
        u_to_u_delay=200.0,
        polarity='pp',
        base_offset=0.1,
        capture_width_ns=1000.0,
        num_averages=3,
        vdiv=0.2,
        save_directory='test_uduu',
        auto_trigger=True,
        save_data=False,
        save_plot=False,
        verbose=True
    )

Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1074,SN:BNPR740043,SCPI:1999.0,SV:1.3.34.0
Connected to: TEKTRONIX,TDS6604,B010222,CF:91.1CT FV:2.4.0
Polarity: pp, Averages: 3
Scope averaging enabled: 3 acquisitions
Scope: 200.0 ns/div, 10000 points

Average 0/3

Average 1/3

Average 2/3

Captured: 10000 points (averaged over 3)


In [ ]:
pulser = BNC765("TCPIP::169.254.209.156::INSTR")

# setup_pulse_channel(
#     pulser=pulser, channel=1, voltage=0.5,
#     offset=0.1, inverted=True,
#     capture_width_ns=1000,
#     pulses=[{'width_ns': 200, 'delay_ns': 0}], verbose=False
# )

pulser.write(f'OUTPUT1:PULSE:MODE DOU')
pulser.write(f'SOURCE1:PULSE1:WIDTH 200E-9')
pulser.write(f'SOURCE1:PULSE1:DELAY 0')
pulser.write(f'SOURCE1:PULSE2:WIDTH 200E-9')
pulser.write(f'SOURCE1:PULSE2:DELAY 300e-9')
pulser.trigger_output_polarity = 'POSITIVE'



In [ ]:
# """
# UDUU - Four Pulse Protocol for Ferroelectric Measurements
# U-D-U-U pulse sequence with Python-timed delay between UD and UU stages
# """
# import time
# import numpy as np
# import matplotlib.pyplot as plt
# from bnc765_driver import BNC765
# from tds6604_driver import TDS6604
# from pathlib import Path
# from datetime import datetime
#
# def setup_pulse_channel(pulser, channel, voltage, offset, capture_width_ns, pulses,
#                         inverted=False, verbose=False):
#     """Setup pulse channel with minimal output"""
#     num_pulses = len(pulses)
#     if num_pulses < 1 or num_pulses > 4:
#         raise ValueError("Must specify 1-4 pulses")
#     pulser.stop()
#     ch = getattr(pulser, f'ch{channel}')
#     ch.output_state = False
#     # time.sleep(0.3)
#     pulse_mode_map = {1: 'SIN', 2: 'DOU', 3: 'TRI', 4: 'QUAD'}
#     ch.pulse_mode = pulse_mode_map[num_pulses]
#     # time.sleep(0.2)
#     ch.inverted = inverted
#     ch.voltage_level = voltage
#     ch.voltage_offset = offset + voltage / 2 if not inverted else offset - voltage / 2
#     ch.load_impedance = 50
#     period_ns = capture_width_ns
#     ch.period = f'{period_ns}ns'
#     # time.sleep(0.2)
#     for i, pulse in enumerate(pulses):
#         pulse_num = i + 1
#         width_ns = pulse.get('width_ns', 100)
#         delay_ns = pulse.get('delay_ns', 0)
#         pulse_prefix = f'SOURCE{channel}:PULSE{pulse_num}'
#         pulser.write(f'{pulse_prefix}:WIDTH {width_ns}E-9')
#         pulser.write(f'{pulse_prefix}:DELAY {delay_ns}E-9')
#     ch.burst_ncycles = 1
#     pulser.trigger_mode = 'BURST'
#     pulser.trigger_source = 'MANUAL'
#     pulser.trigger_output_amplitude = 0.9
#     pulser.trigger_output_polarity = 'POSITIVE'
#     pulser.write(f'TRIGGER:OUTPUT:SOURCE OUT{channel}')
#     ch.output_state = True
#     pulser.start()
#     if verbose:
#         print(f"CH{channel}: {num_pulses} pulse(s), {voltage}V")
#
# def setup_scope(scope, signal_channel, trigger_channel, max_voltage,
#                 capture_width_ns, record_length, vdiv, num_averages=1, verbose=False):
#     """Setup oscilloscope with averaging"""
#     sig_ch = getattr(scope, f'ch{signal_channel}')
#     sig_ch.enabled = True
#     sig_ch.coupling = 'DC'
#     sig_ch.impedance = 'FIFTY'
#     sig_ch.scale = vdiv
#     sig_ch.position = 0
#     trig_ch = getattr(scope, f'ch{trigger_channel}')
#     trig_ch.enabled = True
#     trig_ch.coupling = 'DC'
#     trig_ch.impedance = 'FIFTY'
#     trig_ch.scale = 0.25
#     trig_ch.position = 0
#     scope.record_length = record_length
#     scope.timebase = capture_width_ns * 1e-9 / 10
#     scope.setup_edge_trigger(
#         source=f'CH{trigger_channel}',
#         level=0.45,
#         slope='RISE',
#         mode='NORMAL'
#     )
#     scope.horizontal_position = 0
#     if num_averages > 1:
#         scope.acquisition_mode = 'AVERAGE'
#         scope.write(f'ACQUIRE:NUMAVG {num_averages}')
#         if verbose:
#             print(f"Scope averaging enabled: {num_averages} acquisitions")
#     else:
#         scope.acquisition_mode = 'SAMPLE'
#     scope.acquisition_stopafter = 'SEQUENCE'
#     if verbose:
#         print(f"Scope: {scope.timebase * 1e9:.1f} ns/div, {record_length} points")
#
# def print_timing_report(markers, target_delay_s):
#     """Print detailed timing analysis"""
#     print("\n" + "="*60)
#     print("TIMING ANALYSIS")
#     print("="*60)
#
#     ref = markers['start']
#     for key, timestamp in markers.items():
#         print(f"{key:.<40} {(timestamp - ref)*1e3:.3f} ms")
#
#
# def run_uduu(pulse_amplitude, pulse_width_ns, u_to_d_delay, d_to_u_delay, u_to_u_delay,
#              polarity='uu', base_offset=0.0, capture_width_ns=2000.0, vdiv=0.5,
#              record_length=10000, save_directory=None, auto_trigger=False,
#              save_plot=False, save_data=True, verbose=False):
#     """
#     Run UDUU 4-Pulse Protocol measurement with Python-timed delay
#     """
#     if polarity not in ['nn', 'pp']:
#         raise ValueError("polarity must be 'nn' or 'pp'")
#
#     markers = {}
#     markers['start'] = time.perf_counter()
#
#     pulser = BNC765("TCPIP::169.254.125.69::INSTR")
#     scope = TDS6604('GPIB0::2::INSTR')
#
#     try:
#         markers['instruments_connected'] = time.perf_counter()
#
#         # Calculate pulse timing
#         d_delay = pulse_width_ns + u_to_d_delay
#
#         ud_period_ns = 2 * pulse_width_ns + u_to_d_delay + 50  # 50ns margin
#         uu_period_ns = 2 * pulse_width_ns + u_to_u_delay + 50  # 50ns margin
#
#         # Determine polarity
#         if polarity == 'pp':
#             u_inverted = False
#             d_inverted = True
#         else:  # 'nn'
#             u_inverted = True
#             d_inverted = False
#
#         u_offset = base_offset
#         d_offset = base_offset
#
#         # ========== STEP 1: Load UD Pulses ==========
#         markers['ud_load_start'] = time.perf_counter()
#
#         pulses_u1 = [{'width_ns': pulse_width_ns, 'delay_ns': 0}]  # U pulse
#         pulses_d = [{'width_ns': pulse_width_ns, 'delay_ns': d_delay}]  # D pulse
#
#         setup_pulse_channel(
#             pulser=pulser, channel=1, voltage=pulse_amplitude,
#             offset=u_offset, inverted=u_inverted,
#             capture_width_ns=ud_period_ns,
#             pulses=pulses_u1, verbose=verbose
#         )
#
#         setup_pulse_channel(
#             pulser=pulser, channel=2, voltage=pulse_amplitude,
#             offset=d_offset, inverted=d_inverted,
#             capture_width_ns=ud_period_ns,
#             pulses=pulses_d, verbose=verbose
#         )
#
#         markers['ud_load_complete'] = time.perf_counter()
#
#         # ========== STEP 2: Load Scope Settings ==========
#         markers['scope_load'] = time.perf_counter()
#
#         setup_scope(
#             scope=scope,
#             signal_channel=1,
#             trigger_channel=4,
#             max_voltage=pulse_amplitude,
#             capture_width_ns=ud_period_ns,
#             record_length=record_length,
#             vdiv=vdiv,
#             verbose=verbose
#         )
#
#         # Calculate before the timing-critical section
#         u2_delay = 5e-9
#         u3_delay = (pulse_width_ns + u_to_u_delay + 5)*1e-9
#         scope.timebase = capture_width_ns * 1e-9 / 10
#         target_delay_s = d_to_u_delay * 1e-9
#
#         # ========== STEP 3: Trigger Pulser (UD) ==========
#         if not auto_trigger:
#             input("Press Enter to trigger U-D sequence...")
#
#         markers['ud_trigger'] = time.perf_counter()
#         pulser.trigger()
#
#         # Start scope arming (non-blocking if possible)
#
#         markers['scope_arm_start'] = time.perf_counter()
#         scope.arm()
#
#         # Reconfigure pulser in parallel
#         markers['uu_load_start'] = time.perf_counter()
#         pulser.ch2.output_state = False
#         pulser.write(f'OUTPUT1:PULSE:MODE DOU')
#         pulser.write(f'SOURCE1:PULSE1:WIDTH {pulse_width_ns}E-9')
#         pulser.write(f'SOURCE1:PULSE1:DELAY {u2_delay}')
#         pulser.write(f'SOURCE1:PULSE2:WIDTH {pulse_width_ns}E-9')
#         pulser.write(f'SOURCE1:PULSE2:DELAY {u3_delay}E-9')
#         markers['uu_load_complete'] = time.perf_counter()
#         markers['scope_armed'] = time.perf_counter()
#
#
#         markers['uu_load_complete'] = time.perf_counter()
#
#         # ========== PYTHON DELAY ==========
#         markers['python_delay_start'] = time.perf_counter()
#
#         time.sleep(target_delay_s-0.55)
#
#         # ========== STEP 6: Trigger UU Pulses ==========
#         markers['uu_trigger'] = time.perf_counter()
#         pulser.trigger()
#
#         # Wait for scope capture
#         if scope.wait_for_trigger(timeout=5):
#             markers['acquisition_complete'] = time.perf_counter()
#
#             ch1_data = scope.ch1.get_waveform()
#
#             if verbose:
#                 print(f"Captured: {len(ch1_data['voltage'])} points")
#                 print(f"UD period: {ud_period_ns:.1f} ns")
#                 print(f"UU period: {uu_period_ns:.1f} ns")
#                 print_timing_report(markers, target_delay_s)
#
#             # Save data...
#             # (rest of your save code)
#
#             return ch1_data, markers
#         else:
#             raise TimeoutError("Scope timeout - no trigger received")
#
#     finally:
#         markers['cleanup_start'] = time.perf_counter()
#         pulser.ch1.output_state = False
#         pulser.ch2.output_state = False
#         pulser.stop()
#         pulser.shutdown()
#         scope.shutdown()
#         markers['cleanup_complete'] = time.perf_counter()
#
# # Example usage
# if __name__ == "__main__":
#     data, timing_markers = run_uduu(
#         pulse_amplitude=0.5,
#         pulse_width_ns=500.0,
#         u_to_d_delay=500.0,
#         d_to_u_delay=1e9,  # 1s
#         u_to_u_delay=500.0,
#         polarity='pp',
#         capture_width_ns=1000.0,
#         save_directory='test_uduu',
#         auto_trigger=True,
#         save_data=False,
#         save_plot=False,
#         verbose=True,
#     )

In [ ]:
from bnc765_driver import BNC765
pulser = BNC765("TCPIP::169.254.125.69::INSTR")
# pulser.write('SOUR1:PUlSE2:DELAY 400e-9')
# pulser.write('SOUR1:PULSE2:DELAY 500e-9')
pulser.ch2.voltage_level=0
# pulser.trigger_output_polarity = 'POS'
pulser.ch2.pulse_delay = 700*1e-9

In [ ]:
    for avg_num in range(num_averages):
         print(f"\nAverage {avg_num + 1}/{num_averages}")

In [4]:
from tds6604_driver import TDS6604
scope = TDS6604('GPIB0::1::INSTR')
scope.write(f'ACQUIRE:NUMAVG 1')
num_avg = scope.acquisition_numavg
num_avg

2.0